# 🎯 Step 3: Model Evaluation & Visual Inference
Testing the trained model on unseen data to verify real-world performance.

In [ ]:
import os
import cv2
import random
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path
from IPython.display import Image, display

BASE_DIR = Path(os.getcwd()).absolute()
MODEL_PATH = BASE_DIR / 'saved_models' / 'best.pt'
TEST_IMG_DIR = BASE_DIR / 'HelmetDataset' / 'test' / 'images'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

model = YOLO(str(MODEL_PATH))
print(f"✅ Model loaded from {MODEL_PATH}")

In [ ]:
# 1. Quantitative Evaluation
metrics = model.val(data=str(BASE_DIR / 'HelmetDataset' / 'data.yaml'), split='test')
print(f"mAP@50: {metrics.box.map50:.4f} | mAP@50-95: {metrics.box.map:.4f}")

# Display training and validation evaluation charts
# Note: YOLOv8 saves results in 'runs/detect/helmet_detect' based on the new train_model.ipynb setup
charts_dir = BASE_DIR / 'runs' / 'detect' / 'helmet_detect'
charts = [
    charts_dir / 'confusion_matrix.png',
    charts_dir / 'BoxPR_curve.png',
    charts_dir / 'BoxF1_curve.png'
]

for chart in charts:
    if chart.exists():
        img = cv2.imread(str(chart))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(8, 6))
            plt.imshow(img)
            plt.title(chart.name, fontsize=13)
            plt.axis('off')
            plt.show()
        else:
            print(f"⚠️ Could not read image: {chart.name}")
    else:
        print(f"ℹ️ Chart not found: {chart.name}")


In [ ]:
# 2. Visual Inference on Random Test Samples
images = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
samples = random.sample(images, min(9, len(images)))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i, img_name in enumerate(samples):
    res = model.predict(str(TEST_IMG_DIR / img_name), conf=0.3, verbose=False)[0]
    annotated = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    axes[i].imshow(annotated)
    axes[i].set_title(img_name, fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()